# 01 · Data Exploration

First look at the collected sources and annotated facts. Reproduces the EDA tables in `docs/data/phase2_report.md`.

Run the pipeline first (`pipelines/collect_sources.py`, `extract_text.py`, `coverage.py`).

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('..') / 'data'
services = pd.read_csv(DATA / 'metadata' / 'services.csv')
sources = pd.read_csv(DATA / 'metadata' / 'sources.csv')
service_sources = pd.read_csv(DATA / 'metadata' / 'service_sources.csv')
log = pd.read_csv(DATA / 'metadata' / 'collection_log.csv')
documents = pd.read_csv(DATA / 'processed' / 'documents.csv')
facts = pd.read_csv(DATA / 'processed' / 'annotations' / 'service_facts.csv', keep_default_na=False)
coverage = pd.read_csv(DATA / 'processed' / 'field_coverage.csv')

## Services and sources
Which kinds of services are in scope, and how trustworthy and in which language are their sources?

In [2]:
display(services.groupby(['category', 'government_level']).size().rename('services'))
display(pd.crosstab(sources['tier'], sources['language'], margins=True))

category              government_level
business              national            2
civil_status          commune             5
identity              national            1
identity_travel       national            2
local_administration  district            1
transport             national            3
visa                  national            1
Name: services, dtype: int64

language,en,km,All
tier,,,
T1,4,14,18
T2,3,0,3
T3,1,0,1
All,8,14,22


In [3]:
# Which services have an official (T1) source that actually states requirements?
linked = service_sources.merge(sources[['source_id', 'tier']], on='source_id')
has_primary = linked[(linked['tier'] == 'T1') & (linked['relevance'] == 'primary')]['service_slug'].unique()
services.assign(has_t1_primary=services['service_slug'].isin(has_primary))[['service_slug', 'category', 'has_t1_primary']]

,service_slug,category,has_t1_primary
0,passport_new,identity_travel,False
1,passport_renewal,identity_travel,False
2,national_id_card,identity,False
3,birth_registration,civil_status,False
4,late_birth_registration,civil_status,False
5,marriage_registration,civil_status,False
6,death_registration,civil_status,False
7,civil_record_correction,civil_status,False
8,driver_license_ab,transport,True
9,driver_license_exchange,transport,True


## Collection outcomes
Latest attempt per source.

In [4]:
latest = log.sort_values('collected_at').groupby('source_id').tail(1)
latest['outcome'].value_counts()

outcome
new               21
blocked_robots     1
Name: count, dtype: int64

## Extracted text: how much Khmer is there?
Pages declared as English still carry Khmer content (FAQs), which matters for extraction and search.

In [5]:
docs = documents.merge(sources[['source_id', 'language', 'tier']], on='source_id')
display(docs.groupby('method').agg(n=('chars', 'size'), median_chars=('chars', 'median'), mean_khmer=('khmer_char_ratio', 'mean')).round(2))
docs.groupby('language')['khmer_char_ratio'].describe().round(2)

,n,median_chars,mean_khmer
method,,,
html,17,5424.0,0.76
ocr,3,2376.0,0.67
pdf_text,1,15422.0,0.00


,count,mean,std,min,25%,50%,75%,max
language,,,,,,,,
en,7.0,0.33,0.42,0.00,0.00,0.00,0.77,0.81
km,14.0,0.90,0.13,0.62,0.92,0.95,0.98,0.98


## Annotated facts and field coverage
What do official sources state, and what do they leave out?

In [6]:
stated = facts[facts['status'] == 'stated']
evidence_lang = stated['evidence'].map(lambda e: 'km' if any('\u1780' <= c <= '\u17ff' for c in e) else 'en')
display(evidence_lang.value_counts().rename('facts by evidence language'))
display(pd.crosstab(facts['service_slug'], facts['field']))
coverage.dropna(subset=['completeness'])

evidence
en    28
km    12
Name: facts by evidence language, dtype: int64

field,condition,contact,eligibility,fee,location,processing_time,required_document,step,validity
service_slug,,,,,,,,,
driver_license_ab,1,1,2,4,2,1,3,3,1
driver_license_exchange,1,1,1,1,1,1,6,0,0
vehicle_registration,0,1,0,2,1,2,4,3,0


,service_slug,eligibility_status,eligibility_count,required_document_status,required_document_count,step_status,step_count,fee_status,fee_count,processing_time_status,processing_time_count,location_status,location_count,contact_status,contact_count,completeness
8,driver_license_ab,stated,2,stated,3,stated,3,stated,4,not_stated,0,stated,2,stated,1,0.857
9,driver_license_exchange,stated,1,stated,6,not_stated,0,stated,1,not_stated,0,stated,1,stated,1,0.714
10,vehicle_registration,not_stated,0,stated,4,stated,3,stated,2,stated,1,stated,1,stated,1,0.857
